# Fig. S1K: Reactome enrichment of the expression-invisible APA genes

- Author: Andre Lopes Martins Macedo
- Date: 2026-10-08
- Purpose: Show which Reactome pathways are over-represented among the 131 genes whose
  3'UTR site usage changes along pseudotime while their expression stays flat, as a ridgeplot
  in the style of Fig. 1F and fig. S1J.

Data: adult mouse SVZ lineage (wild type), Dirichlet-multinomial pseudotime test
(`DirMult_optimized`, df = 5). Foreground = 131 genes with a significant chi-square LRT
(BH adjusted P < 0.05) and |Spearman correlation of expression with pseudotime-bin order| < 0.3;
universe = the 7,028 tested genes. The ORA uses the settings of the original analysis script
(ReactomePA `enrichPathway`, mouse, BH, minGSSize 5); it is recomputed here and checked against
the stored table.

Kernel: R (`apa-r` env). Outputs: `.../paper_2026_ASD/Fig1/supp_12_expression_invisible_reactome/`
and a copy in `.../Draft-v1/Fig1/supp_fig1_12_figS1K_expression_invisible_reactome/`.

## 1. Setup

Load the shared paper style (font sizes, `theme_paper`, `save_and_show()`, panel-size constants),
the enrichment packages, and define paths and panel parameters. The panel is 85 x 58 mm, the
same size as the fig. S1J ridgeplot. Text is set to Arial (the font of the final figure masters;
the shared theme uses the metric-identical Liberation Sans), and svglite keeps it as text.
No step uses random numbers; the seed is set only by convention.

## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
REPO_ROOT <- Sys.getenv("M26_REPO_ROOT")
if (!nzchar(REPO_ROOT)) {
  REPO_ROOT <- normalizePath(getwd())
  while (!file.exists(file.path(REPO_ROOT, "environment", "apa-python.yml")) && REPO_ROOT != dirname(REPO_ROOT)) {
    REPO_ROOT <- dirname(REPO_ROOT)
  }
  Sys.setenv(M26_REPO_ROOT = REPO_ROOT)
}
DATA_ROOT <- Sys.getenv("M26_DATA_ROOT")
if (!nzchar(DATA_ROOT)) stop("Set M26_DATA_ROOT (see README)")
RESULTS_ROOT <- Sys.getenv("M26_RESULTS_ROOT", file.path(REPO_ROOT, "results"))

In [ ]:
source(paste0(REPO_ROOT, "/figures/style/paper_fig_style.R"))
suppressPackageStartupMessages({
  library(clusterProfiler)
  library(ggridges)
  library(org.Mm.eg.db)
  library(ReactomePA)
  library(readr)
  library(stringr)
  library(tibble)
  library(dplyr) # attached last so its verbs are not masked by IRanges / AnnotationDbi
})
set.seed(20261008)

CORR_DIR <- paste0(DATA_ROOT, "/3utr/wildtypes_analysis/2026_analyses/utrcalling/correlation_analyses_proportions")
DM_DIR <- paste0(DATA_ROOT, "/3utr/wildtypes_analysis/2026_analyses/utrcalling/DirMult_optimized")
PANEL <- "supp_12_expression_invisible_reactome"
PANEL_DIR <- file.path(OUT_ROOT, "Fig1", PANEL)
DRAFT_DIR <- file.path(
  OUT_ROOT, "Draft-v1", "Fig1",
  "supp_fig1_12_figS1K_expression_invisible_reactome"
)
FIG_W <- HALF_W_IN # 85 mm, as fig. S1J
FIG_H <- mm_to_in(58) # 58 mm, as fig. S1J
FONT_FAMILY <- "Arial"
PADJ_CUT <- 0.05
MAX_TERMS <- 8
# terms quoted in the Results text (preferred representatives)
RESULTS_TERMS <- c(
  "Cellular responses to stress",
  "APC/C:Cdc20 mediated degradation of mitotic proteins",
  "Metabolism of RNA",
  "Cap-dependent Translation Initiation"
)
stopifnot(nrow(systemfonts::match_fonts(FONT_FAMILY)) == 1)
cat("font file:", systemfonts::match_fonts(FONT_FAMILY)$path, "\n")

## 2. Recompute the Reactome ORA and check it against the stored table

Same calls as the original analysis script: Ensembl IDs are mapped to Entrez with `bitr`, then
`enrichPathway` (mouse, BH, minGSSize 5, all terms returned). The recomputed table must match
`invisible_ora_Reactome.csv` term by term (adjusted P, counts, gene lists), give 30 terms at
adjusted P < 0.05, and reproduce the two values quoted in the Results.

In [ ]:
fg_ens <- read_csv(file.path(CORR_DIR, "invisible_foreground.csv"), show_col_types = FALSE)$gene_id
bg_ens <- read_csv(file.path(CORR_DIR, "invisible_bg_all.csv"), show_col_types = FALSE)$gene_id
stopifnot(length(fg_ens) == 131, length(bg_ens) == 7028, all(fg_ens %in% bg_ens))

bg_map <- bitr(bg_ens, fromType = "ENSEMBL", toType = "ENTREZID", OrgDb = org.Mm.eg.db)
fg_entrez <- bitr(fg_ens, fromType = "ENSEMBL", toType = "ENTREZID", OrgDb = org.Mm.eg.db)$ENTREZID
bg_entrez <- bg_map$ENTREZID

ora <- enrichPathway(
  fg_entrez,
  universe = bg_entrez,
  organism = "mouse",
  pAdjustMethod = "BH",
  pvalueCutoff = 1,
  qvalueCutoff = 1,
  minGSSize = 5,
  readable = TRUE
)
ora_df <- as_tibble(as.data.frame(ora))

stored <- read_csv(file.path(CORR_DIR, "invisible_ora_Reactome.csv"), show_col_types = FALSE)
cmp <- stored |>
  dplyr::select(ID, padj_stored = p.adjust, count_stored = Count, genes_stored = geneID) |>
  inner_join(dplyr::select(ora_df, ID, p.adjust, Count, geneID), by = "ID")
stopifnot(
  nrow(cmp) == nrow(stored),
  nrow(cmp) == nrow(ora_df),
  isTRUE(all.equal(cmp$padj_stored, cmp$p.adjust, tolerance = 1e-8)),
  all(cmp$count_stored == cmp$Count),
  all(cmp$genes_stored == cmp$geneID)
)

sig <- ora_df |>
  filter(p.adjust < PADJ_CUT) |>
  arrange(p.adjust, pvalue)
stress <- filter(sig, Description == "Cellular responses to stress")
cap <- filter(sig, Description == "Cap-dependent Translation Initiation")
stopifnot(
  nrow(sig) == 30,
  signif(stress$p.adjust, 3) == 3.76e-4, stress$Count == 20,
  signif(cap$p.adjust, 3) == 0.0212, cap$Count == 5
)
cat(sprintf(
  "reproduced: %d/%d terms identical; %d terms at adj P < %.2f\n",
  nrow(cmp), nrow(stored), nrow(sig), PADJ_CUT
))
cat(sprintf("Cellular responses to stress: adj P = %.3g, %d genes\n", stress$p.adjust, stress$Count))
cat(sprintf("Cap-dependent Translation Initiation: adj P = %.3g, %d genes\n", cap$p.adjust, cap$Count))

## 3. Term selection: collapse redundant terms, keep the top 8

Rule:

1. Start from the 30 terms with ORA adjusted P < 0.05.
2. Group terms whose foreground gene sets (the hit genes) are identical; each group is shown
   once, by one representative.
3. Representative: the term quoted in the Results text if the group contains one; otherwise the
   smallest term (fewest tested genes, i.e. the most specific); remaining ties go to the shorter name.
4. Order the representatives by their own adjusted P (ties broken by the unadjusted P) and show
   the first 8.

The table below lists every group: its representative, whether it is shown, and the terms
collapsed into it.

In [ ]:
sig <- sig |>
  mutate(
    n_tested = as.integer(str_extract(BgRatio, "^[0-9]+")),
    fg_key = vapply(
      strsplit(geneID, "/"),
      \(g) paste(sort(g), collapse = "/"),
      character(1)
    ),
    set_group = match(fg_key, unique(fg_key)),
    in_results = Description %in% RESULTS_TERMS
  )

reps <- sig |>
  group_by(set_group) |>
  arrange(desc(in_results), n_tested, nchar(Description), .by_group = TRUE) |>
  slice(1) |>
  ungroup() |>
  arrange(p.adjust, pvalue) |>
  mutate(rank = row_number(), shown = rank <= MAX_TERMS)

collapse_tbl <- reps |>
  dplyr::select(rank, set_group, representative = Description, shown, p.adjust, Count, n_tested) |>
  left_join(
    sig |>
      dplyr::select(set_group, Description) |>
      group_by(set_group) |>
      summarise(members = list(Description), .groups = "drop"),
    by = "set_group"
  ) |>
  mutate(
    collapsed = mapply(
      \(m, r) paste(setdiff(m, r), collapse = " | "),
      members, representative
    )
  ) |>
  dplyr::select(-members, -set_group)

stopifnot(all(RESULTS_TERMS %in% reps$Description[reps$shown]))
cat(sprintf(
  "%d significant terms -> %d distinct foreground gene sets; %d shown\n\n",
  nrow(sig), nrow(reps), sum(reps$shown)
))
collapse_tbl |>
  dplyr::select(-collapsed) |>
  mutate(p.adjust = signif(p.adjust, 3)) |>
  as.data.frame() |>
  print(right = FALSE)
cat("\nCollapsed terms (representative <- identical foreground gene set):\n")
with(
  filter(collapse_tbl, collapsed != ""),
  cat(sprintf("  %2d %s%s\n      <- %s\n", rank, representative, ifelse(shown, " [shown]", ""), collapsed), sep = "")
)

## 4. Display labels

Reactome names are kept verbatim except where they are too long for the label column; the one
shortened name keeps its meaning (the APC/C:Cdc20 complex mediates the degradation). Labels are
wrapped (not truncated) at 26 characters, as in fig. S1J, and end with n = number of tested genes
(of the 7,028) annotated to the term.

In [ ]:
label_map <- tribble(
  ~Description, ~short_name,
  "APC/C:Cdc20 mediated degradation of mitotic proteins", "APC/C:Cdc20 degradation of mitotic proteins"
)
selected <- reps |>
  filter(shown) |>
  left_join(label_map, by = "Description") |>
  mutate(
    short_name = coalesce(short_name, Description),
    label = sprintf("%s  (n=%d)", str_wrap(short_name, 26), n_tested)
  )
selected |>
  dplyr::select(rank, Description, short_name, n_tested, Count, p.adjust) |>
  mutate(p.adjust = signif(p.adjust, 3)) |>
  as.data.frame() |>
  print(right = FALSE)

## 5. Per-gene P values of the tested genes in each term

Each ridge shows the distribution of -log10 P of the chi-square likelihood-ratio test
(`p_value_chisq`, the column behind the GSEA ranking used by Fig. 1F and fig. S1J) for all tested
genes annotated to the term, not only the foreground hits. Values come from
`composite_ranking_optimized2_df5.csv` (the 7,028 tested genes; identical to the LRT table for
these genes). As in the earlier panels, P = 0 is clamped to the smallest non-zero P. Gene-to-term
membership is the Reactome annotation used by the ORA, restricted to the ORA universe, so n equals
the BgRatio numerator of the ORA table (asserted).

In [ ]:
dm <- read_csv(
  file.path(DM_DIR, "composite_ranking_optimized2_df5.csv"),
  col_select = c(gene_id, gene_name, p_value_chisq),
  show_col_types = FALSE
)
stopifnot(setequal(dm$gene_id, bg_ens), !anyNA(dm$p_value_chisq))
min_p <- min(dm$p_value_chisq[dm$p_value_chisq > 0])
dm <- mutate(dm, neglog10p = -log10(pmax(p_value_chisq, min_p)))

gene_p <- bg_map |>
  inner_join(dm, by = c("ENSEMBL" = "gene_id")) |>
  group_by(ENTREZID) |>
  summarise(neglog10p = max(neglog10p), n_ensembl = n_distinct(ENSEMBL), .groups = "drop")

universe_entrez <- unique(ora@universe)
df <- bind_rows(lapply(seq_len(nrow(selected)), \(i) {
  ids <- intersect(ora@geneSets[[selected$ID[i]]], universe_entrez)
  tibble(
    term = selected$Description[i],
    label = selected$label[i],
    entrez = ids,
    neglog10p = gene_p$neglog10p[match(ids, gene_p$ENTREZID)],
    n = length(ids),
    ora_padj = selected$p.adjust[i]
  )
}))

n_check <- df |>
  distinct(term, n) |>
  left_join(dplyr::select(selected, term = Description, n_tested), by = "term")
stopifnot(!anyNA(df$neglog10p), all(n_check$n == n_check$n_tested))
n_ambiguous <- sum(gene_p$n_ensembl[match(unique(df$entrez), gene_p$ENTREZID)] > 1)
cat(sprintf(
  "%d gene-term rows; %d distinct genes; %d with >1 Ensembl ID (max -log10 P used)\n",
  nrow(df), n_distinct(df$entrez), n_ambiguous
))
cat(sprintf(
  "share of per-gene values beyond the x range (> 12.5): %.1f%%\n",
  100 * mean(df$neglog10p > 12.5)
))

df <- df |>
  mutate(
    label = factor(label, levels = rev(selected$label)),
    database = factor("Reactome")
  )

## 6. Ridgeplot

Code and styling follow the fig. S1J / Fig. 1F ridgeplots: density bandwidth 0.5, x range 0 to
12.5, ridge fill = ORA adjusted P on a log10 colour scale (same yellow-to-dark-red palette, limits
= range of the shown terms), dashed line at -log10(0.05), database strip on the left, no in-panel
title. The panel is saved as SVG (text kept as text) and PNG (300 dpi) with `save_and_show()`, then
both files are copied to the Draft-v1 fig. S1 folder.

In [ ]:
brks <- signif(
  exp(seq(log(min(df$ora_padj)), log(max(df$ora_padj)), length.out = 4)),
  1
)
# keep the rounded end breaks inside the limits so every break is labelled
round_down_1sig <- \(x) floor(x / 10^floor(log10(x))) * 10^floor(log10(x))
brks[brks > max(df$ora_padj)] <- round_down_1sig(max(df$ora_padj))
p <- ggplot(df, aes(x = neglog10p, y = label, fill = ora_padj)) +
  geom_density_ridges(
    scale = 1.3,
    alpha = 0.92,
    color = "grey35",
    linewidth = 0.25,
    rel_min_height = 0.008,
    bandwidth = 0.5
  ) +
  facet_grid(database ~ ., switch = "y") +
  scale_fill_gradientn(
    colours = rev(c("#ffffcc", "#fed976", "#fd8d3c", "#e31a1c", "#800026")),
    trans = "log10",
    limits = range(df$ora_padj),
    breaks = brks,
    labels = formatC(brks, format = "g", digits = 1),
    name = expression(ORA * " " * italic(p)[adj]),
    guide = guide_colourbar(
      barwidth = 0.45,
      barheight = 3.5,
      ticks.colour = "grey30"
    )
  ) +
  geom_vline(
    xintercept = -log10(0.05),
    linetype = "dashed",
    color = "black",
    linewidth = 0.4
  ) +
  labs(
    x = expression("-" * "log"[10] * " " * italic(p) * "-value  (per gene in set)"),
    y = NULL
  ) +
  scale_x_continuous(
    expand = expansion(mult = c(0, 0.02)),
    breaks = seq(0, 12.5, 2.5)
  ) +
  # headroom so the top ridge is not clipped and the bottom label clears the x axis (cf. the Fig. 1F variant)
  scale_y_discrete(expand = expansion(add = c(0.35, 1.4))) + # top >= ridge scale (1.3)
  coord_cartesian(xlim = c(0, 12.5), clip = "on") +
  theme_paper +
  theme(
    text = element_text(family = FONT_FAMILY),
    axis.text.y = element_text(size = FONT_LEGEND, lineheight = 0.85),
    legend.position = "right",
    panel.border = element_rect(linewidth = 0.3, fill = NA),
    panel.grid.major.x = element_line(linewidth = 0.25, color = "gray90"),
    panel.grid.major.y = element_blank(),
    strip.text.y.left = element_text(size = FONT_TICK, face = "bold", angle = 90),
    strip.background = element_rect(fill = "#F0F0F0", linewidth = 0.4),
    strip.placement = "outside",
    panel.spacing.y = unit(2, "pt")
  )

save_and_show(p, PANEL, FIG_W, FIG_H)
cat("colour breaks:", formatC(brks, format = "g", digits = 1), "\n")
cat(sprintf("colour limits: %.3g to %.3g\n", min(df$ora_padj), max(df$ora_padj)))

## 7. Check the SVG and copy the panel to the Draft-v1 fig. S1 folder

Confirm that the SVG keeps every label as a text element in Arial (no glyph outlines), then copy
the SVG and PNG, plus a small CSV with the shown terms, labels and collapsed terms (for the legend).

In [ ]:
svg_path <- file.path(PANEL_DIR, paste0(PANEL, ".svg"))
png_path <- file.path(PANEL_DIR, paste0(PANEL, ".png"))
svg_txt <- readLines(svg_path, warn = FALSE)
n_text <- sum(str_count(svg_txt, "<text"))
fonts_used <- unique(unlist(str_extract_all(svg_txt, "font-family: [^;']+")))
stopifnot(n_text > 0, all(str_detect(fonts_used, "Arial")), !any(str_detect(svg_txt, "glyphgroup'>")))
cat(sprintf("SVG: %d text elements; fonts: %s\n", n_text, paste(fonts_used, collapse = ", ")))

terms_out <- selected |>
  dplyr::select(rank, ID, term = Description, label = short_name, n_tested, n_foreground = Count, pvalue, p.adjust) |>
  left_join(
    dplyr::select(collapse_tbl, term = representative, collapsed_terms = collapsed),
    by = "term"
  )
csv_path <- file.path(PANEL_DIR, paste0(PANEL, "_terms.csv"))
write_csv(terms_out, csv_path)

dir.create(DRAFT_DIR, recursive = TRUE, showWarnings = FALSE)
copied <- file.copy(c(svg_path, png_path, csv_path), DRAFT_DIR, overwrite = TRUE)
stopifnot(all(copied))
print(list.files(DRAFT_DIR, full.names = TRUE))

## Summary

- ORA reproduced: the recomputed Reactome ORA (131 foreground genes, 7,028-gene universe)
  matches `invisible_ora_Reactome.csv` term by term; 30 terms at adjusted P < 0.05; Cellular
  responses to stress adjusted P = 3.76e-4 (20 genes), Cap-dependent Translation Initiation 0.0212
  (5 genes).
- Panel: 8 non-redundant terms (identical foreground gene sets collapsed, Section 3), ordered by
  ORA adjusted P. Ridges = density of -log10 `p_value_chisq` (chi-square LRT of the
  Dirichlet-multinomial pseudotime test) of all tested genes in the term; fill = ORA adjusted P
  (log scale); n = tested genes in the term; dashed line, P = 0.05.
- Files:
  - `$M26_RESULTS_ROOT/Fig1/supp_12_expression_invisible_reactome/`
    (`.svg`, `.png` at 300 dpi, `_terms.csv`)
  - copies in `$M26_RESULTS_ROOT/Draft-v1/Fig1/supp_fig1_12_figS1K_expression_invisible_reactome/`